In [1]:
# Проверка задания п2.1.6 на устройстве: читает таблицу векторов и регистр GPIO при разном состоянии светодиода
# и записывает обмен в файл device-2-1-6.log.

import time
from datetime import datetime

import serial
from serial.tools import list_ports

VENDOR_ID = 0x2E8A
PRODUCT_ID = 0x000A

TASK = "2.1.6"
PROJECT = "211-command-usb"
LOG_NAME = "device-2-1-6.log"
COMMANDS = ["boot_info", "enable", "boot_info", "disable", "boot_info"]
ANSWER_S = 2


def find_board():
    for port in list_ports.comports():
        if port.vid == VENDOR_ID and port.pid == PRODUCT_ID:
            return port
    return None


def talk(board):
    exchange = []
    with serial.Serial(board.device, timeout=0.5) as port:
        time.sleep(0.2)
        port.reset_input_buffer()
        started = time.monotonic()
        for command in COMMANDS:
            port.write((command + "\n").encode("ascii"))
            exchange.append((time.monotonic() - started, "-->", command))
            deadline = time.monotonic() + ANSWER_S
            while time.monotonic() < deadline:
                line = port.readline().decode("ascii", "replace").strip()
                if line:
                    exchange.append((time.monotonic() - started, "<--", line))
                    print(line, end="\r\n")
    return exchange


def write_log(board, exchange):
    with open(LOG_NAME, "w", encoding="utf-8") as log:
        log.write("задание: " + TASK + "\n")
        log.write("проект: " + PROJECT + "\n")
        log.write("устройство: %04x:%04x\n" % (board.vid, board.pid))
        log.write("серийный номер: " + str(board.serial_number) + "\n")
        log.write("порт: " + board.device + "\n")
        log.write("начало: " + datetime.now().isoformat(timespec="seconds") + "\n")
        for moment, direction, text in exchange:
            log.write("%8.3f %s %s\n" % (moment, direction, text))
        answers = len(exchange) - len(COMMANDS)
        log.write("итог: отправлено команд %d, принято строк %d\n" % (len(COMMANDS), answers))


board = find_board()

if board is None:
    print("Плата не найдена. Проверьте кабель и запишите на плату прошивку задания.", end="\r\n")
else:
    print("Плата на порту " + board.device + ", отправляю команды", end="\r\n")
    exchange = talk(board)
    write_log(board, exchange)
    print("Обмен записан в " + LOG_NAME, end="\r\n")


Плата на порту COM3, отправляю команды
boot_info
dbg read_line:120 got boot_info
vector table   0x10000100
stack top    0x20042000
reset        0x100001f7
reset (even) 0x100001f6
gpio in        0xd0000004
led bit      0
gpio_get     0
enable
dbg read_line:120 got enable
inf cmd_enable:29 led on
boot_info
dbg read_line:120 got boot_info
vector table   0x10000100
stack top    0x20042000
reset        0x100001f7
reset (even) 0x100001f6
gpio in        0xd0000004
led bit      1
gpio_get     1
disable
dbg read_line:120 got disable
inf cmd_disable:35 led off
boot_info
dbg read_line:120 got boot_info
vector table   0x10000100
stack top    0x20042000
reset        0x100001f7
reset (even) 0x100001f6
gpio in        0xd0000004
led bit      0
gpio_get     0
Обмен записан в device-2-1-6.log
